# Bài 2 · Giải Tic-Tac-Toe bằng AND-OR Tree Search

Mỗi nút OR chọn một nước cho tác tử. Với mỗi nước, `results` sinh tập trạng thái sau mọi phản hồi hợp lệ của đối thủ. Nếu nước của tác tử đã kết thúc ván, tập này chỉ gồm trạng thái vừa tạo. `and_search` đòi hỏi tất cả trạng thái trong tập đều có kế hoạch tiếp nối hợp lệ. Bàn cờ có thêm một quân ở mỗi nước nên cây không có chu trình.

`and_or_search` trả về `(nước đi, các nhánh)` cho trạng thái đang chơi. Mỗi nhánh lưu cặp `(trạng thái sau phản hồi, kế hoạch tiếp theo)`. Lá là `win` hoặc `draw`, tùy mục tiêu. `None` nghĩa là không có kế hoạch bảo đảm mục tiêu. `or_search` lưu kết quả theo bộ ba bàn cờ, vai chơi và chính sách chấp nhận hòa để tái sử dụng khi một trạng thái xuất hiện qua nhiều thứ tự đi quân.

In [1]:
from collections import Counter

from and_or_tictactoe import and_or_player, and_or_search, or_search, results
from lab05_experiments import benchmark
from tictactoe import actions, empty_board, other, play, utility


def leaf_outcomes(state, plan, player):
    """Verify that every legal reply is present and count the plan leaves."""
    if isinstance(plan, str):
        assert utility(state, player) == {'win': 1, 'draw': 0}[plan]
        return Counter([plan])
    action, branches = plan
    assert action in actions(state)
    assert tuple(position for position, _ in branches) == results(state, action, player)
    outcomes = Counter()
    for position, child in branches:
        outcomes.update(leaf_outcomes(position, child, player))
    return outcomes


def show_plan(board, plan, player):
    if plan is None:
        print('  Không có kế hoạch bảo đảm mục tiêu.')
        return
    move, branches = plan
    for outcome, child in branches:
        reply = next((i for i in actions(board) if outcome[i] == other(player)), None)
        event = 'ván kết thúc' if reply is None else f'đối thủ đi {reply}'
        continuation = {'win': 'thắng', 'draw': 'hòa'}[child] if isinstance(child, str) else f'đi {child[0]}'
        print(f'  Nếu {event}, kế hoạch ghi {continuation}.')
    outcomes = leaf_outcomes(board, plan, player)
    print(f'  Chọn ô {move}. Các lá thắng = {outcomes["win"]}, hòa = {outcomes["draw"]}.')

## Bốn trạng thái trong notebook mẫu

Các ô trống hiện bằng dấu `.`. Thứ tự chỉ số từ `0` đến `8`, đi từ trái sang phải và từ trên xuống. Hai chế độ lần lượt yêu cầu X không thua và yêu cầu X bắt buộc thắng. Mã kiểm tra rằng mỗi kế hoạch bao phủ tất cả phản hồi của O và rằng lợi ích ở lá khớp với kết quả báo cáo.

In [2]:
cases = {
    'X sắp thắng': list('xo ox    '),
    'X cần tìm đường hòa': list('xox o    '),
    'Tình huống bất lợi': list('o x     o'),
    'Bàn cờ rỗng': empty_board(),
}

for label, board in cases.items():
    print(f'\n{label}')
    for row in range(0, 9, 3):
        print(' '.join(mark if mark != ' ' else '.' for mark in board[row:row + 3]))
    for draw_is_win in (True, False):
        print('Chấp nhận hòa' if draw_is_win else 'Chỉ chấp nhận thắng')
        plan = and_or_search(board, draw_is_win=draw_is_win)
        show_plan(board, plan, 'x')

immediate_win = cases['X sắp thắng']
outcomes = results(immediate_win, 8)
assert len(outcomes) == 1 and utility(outcomes[0], 'x') == 1
assert and_or_search(cases['X cần tìm đường hòa'])[0] == 7
assert and_or_search(cases['X cần tìm đường hòa'], draw_is_win=False) is None
assert and_or_search(cases['Tình huống bất lợi']) is None
assert and_or_search(cases['Bàn cờ rỗng']) is not None
assert and_or_search(cases['Bàn cờ rỗng'], draw_is_win=False) is None


X sắp thắng
x o .
o x .
. . .
Chấp nhận hòa
  Nếu đối thủ đi 5, kế hoạch ghi đi 6.
  Nếu đối thủ đi 6, kế hoạch ghi đi 5.
  Nếu đối thủ đi 7, kế hoạch ghi đi 5.
  Nếu đối thủ đi 8, kế hoạch ghi đi 5.
  Chọn ô 2. Các lá thắng = 5, hòa = 2.
Chỉ chấp nhận thắng
  Nếu đối thủ đi 5, kế hoạch ghi đi 6.
  Nếu đối thủ đi 6, kế hoạch ghi đi 8.
  Nếu đối thủ đi 7, kế hoạch ghi đi 5.
  Nếu đối thủ đi 8, kế hoạch ghi đi 6.
  Chọn ô 2. Các lá thắng = 5, hòa = 0.

X cần tìm đường hòa
x o x
. o .
. . .
Chấp nhận hòa
  Nếu đối thủ đi 3, kế hoạch ghi đi 5.
  Nếu đối thủ đi 5, kế hoạch ghi đi 3.
  Nếu đối thủ đi 6, kế hoạch ghi đi 3.
  Nếu đối thủ đi 8, kế hoạch ghi đi 3.
  Chọn ô 7. Các lá thắng = 3, hòa = 5.
Chỉ chấp nhận thắng
  Không có kế hoạch bảo đảm mục tiêu.

Tình huống bất lợi
o . x
. . .
. . o
Chấp nhận hòa
  Không có kế hoạch bảo đảm mục tiêu.
Chỉ chấp nhận thắng
  Không có kế hoạch bảo đảm mục tiêu.

Bàn cờ rỗng
. . .
. . .
. . .
Chấp nhận hòa
  Nếu đối thủ đi 1, kế hoạch ghi đi 2.
  Nếu đ

- **X sắp thắng.** Ô 8 thắng ngay. Duyệt theo chỉ số tăng dần gặp ô 2 trước. Nước này tạo hai đường chéo cùng dọa thắng ở 6 và 8. O chỉ chặn được một đường nên X vẫn bảo đảm thắng.
- **X cần tìm đường hòa.** X phải đi ô 7 để chặn cột giữa `1–4–7`. Sau đó nhánh tiếp theo phụ thuộc nước O. O có phản hồi giữ được hòa nên mục tiêu bắt buộc thắng không khả thi.
- **Tình huống bất lợi.** Notebook mẫu giả định X tới lượt dù O có hai quân và X mới có một quân. X cần chặn đường chéo `0–4–8` tại ô 4. O đáp ô 6, chặn đường chéo `2–4–6` của X và cùng lúc dọa thắng ở 3 và 7. X chỉ chặn được một ô.
- **Bàn cờ rỗng.** X có thể bảo đảm hòa hoặc thắng nếu O sai. Có ít nhất một nhánh hòa, vì vậy việc tìm kế hoạch luôn thắng thất bại.

## Đấu với Random Agent

Tác tử đi theo kế hoạch đầu tiên bảo đảm không thua. Nếu gặp trạng thái không thể tránh thua khi đối thủ chơi tối ưu, nó chọn ngẫu nhiên trong các nước hợp lệ. Thử nghiệm đặt lại bộ nhớ đệm trước khi bắt đầu và dùng cấu hình hạt giống chung của Lab 05. Thời gian chỉ đo lời gọi tác tử AND-OR.

In [3]:
or_search.cache_clear()
scores = benchmark(and_or_player, games_per_role=100)
print('Vai    Ván    Thắng    Hòa    Thua    ms/nước')
for role, values in [*scores['by_role'].items(), ('Cả hai', scores)]:
    print(f"{role:6} {values['games']:4} {values['wins']:8} {values['draws']:6}"
          f" {values['losses']:7} {values['ms_per_move']:11.3f}")
assert scores['games'] == 200 and scores['losses'] == 0
print('AND-OR đấu AND-OR:', play(and_or_player, and_or_player, N=1))

Vai    Ván    Thắng    Hòa    Thua    ms/nước
x       100       93      7       0       0.046
o       100       74     26       0       0.320
Cả hai  200      167     33       0       0.185
AND-OR đấu AND-OR: {'x': 0, 'o': 0, 'd': 1}


AND-OR giữ được bất bại ở cả hai vai vì luôn chọn một nhánh mà tất cả phản hồi của đối thủ còn giữ được mục tiêu. Tỷ lệ thắng phụ thuộc cách đối thủ chọn nước và thứ tự duyệt. Do cả hòa và thắng được chấp nhận, tác tử có thể lấy nước bảo đảm hòa dù còn nước bảo đảm thắng. Khi cả hai bên theo chính sách này, ván đấu kết thúc hòa.